# Import libraries

In [1]:
!apt-get update

!apt-get install -y \
    tesseract-ocr \
    tesseract-ocr-vie \
    poppler-utils \
    build-essential

!pip install -q \
    pillow \
    pytesseract \
    pdf2image \
    rank_bm25 \
    tqdm \
    numpy \
    scikit-learn \
    torch \
    transformers \
    bitsandbytes \
    sentencepiece \
    accelerate \
    sentence-transformers \
    openai \
    opencv-python-headless \
    colpali-engine

print("All dependencies installed.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,479 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,413 kB]
Get:14 http://archiv

# Load data from Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
import os
import json
from pathlib import Path
from PIL import Image
import logging
import re
from tqdm.notebook import tqdm  # Dùng tqdm.notebook cho Colab
import tempfile
import shutil
import pickle
from typing import List, Dict, Any

# Imports cho các pipeline
from pdf2image import convert_from_path
import pytesseract
from rank_bm25 import BM25Okapi
import torch
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from sentence_transformers import SentenceTransformer, util as st_util
from openai import OpenAI
import getpass
from google.colab import files  # Để upload file

# Thiết lập logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Biến toàn cục
PDF_DIR = Path("/content/drive/MyDrive/CapstoneProject/Retriever/data/pdf")
OUTPUT_DIR = Path("/content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output")

PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OCR_DATA_PATH = OUTPUT_DIR / "ocr_data.json"
BM25_INDEX_PATH = OUTPUT_DIR / "bm25_index.pkl"
COLQWEN_INDEX_PATH = OUTPUT_DIR / "colqwen_index.pkl"
MINILM_INDEX_PATH = OUTPUT_DIR / "minilm_index.pkl"

GLOBAL_OCR_DATA = [] # Sẽ chứa dữ liệu OCR

print(f"Sẽ đọc PDF từ: {PDF_DIR}")
print(f"Sẽ lưu trữ index tại: {OUTPUT_DIR}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")

Sẽ đọc PDF từ: /content/drive/MyDrive/CapstoneProject/Retriever/data/pdf
Sẽ lưu trữ index tại: /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output
Đang sử dụng thiết bị: cuda


# OCR Processor

In [4]:
class SlideOCR:
    """
    Lớp này xử lý việc chuyển đổi PDF -> Ảnh -> Text (OCR).
    """
    def __init__(self):
        # Không cần cài đặt phức tạp, Colab đã có poppler và tesseract
        logging.info("SlideOCR đã sẵn sàng.")

    def pdf_to_images(self, pdf_path: Path, temp_dir: Path) -> List[Path]:
        try:
            images = convert_from_path(
                pdf_path,
                dpi=200,
                output_folder=str(temp_dir),
                fmt="png",
                thread_count=4,
            )
            return [Path(img.filename) for img in images]
        except Exception as e:
            logging.error(f"[SlideOCR] Lỗi khi chuyển đổi PDF {pdf_path}: {e}")
            return []

    def ocr_image(self, image_path: Path) -> str:
        try:
            img = Image.open(image_path)
            # Dùng cả tiếng Việt (vie) và tiếng Anh (eng)
            text = pytesseract.image_to_string(img, lang='vie+eng')
            return text
        except Exception as e:
            logging.error(f"[SlideOCR] Lỗi OCR ảnh {image_path}: {e}")
            return ""

    def process_slides(self, pdf_files: List[Path]) -> List[Dict[str, Any]]:
        all_ocr_data = []

        for pdf_path in tqdm(pdf_files, desc="Đang xử lý PDF"):
            if not pdf_path.exists():
                logging.warning(f"[SlideOCR] File không tồn tại: {pdf_path}")
                continue

            with tempfile.TemporaryDirectory() as temp_dir:
                image_paths = self.pdf_to_images(pdf_path, Path(temp_dir))

                for i, img_path in enumerate(tqdm(image_paths, desc=f"OCR {pdf_path.name}", leave=False)):
                    page_num = i + 1
                    ocr_text = self.ocr_image(img_path)

                    # Chuẩn hóa text
                    clean_text = re.sub(r'\n+', '\n', ocr_text).strip()
                    clean_text = re.sub(r'[ \t]+', ' ', clean_text)

                    if clean_text: # Chỉ lưu nếu có text
                        all_ocr_data.append({
                            "source": pdf_path.name,
                            "page": page_num,
                            "text": clean_text
                        })

        return all_ocr_data

In [5]:
# Nếu muốn chạy lại OCR (tốn thời gian), đặt cờ này thành True
force_rebuild_ocr = False

if not force_rebuild_ocr and OCR_DATA_PATH.exists():
    print(f"Đã tìm thấy file OCR tại: {OCR_DATA_PATH}. Đang tải...")
    with open(OCR_DATA_PATH, 'r', encoding='utf-8') as f:
        GLOBAL_OCR_DATA = json.load(f)
    print(f"Tải {len(GLOBAL_OCR_DATA)} trang từ file JSON thành công.")
else:
    print("Bắt đầu quá trình OCR (việc này có thể mất thời gian)...")
    ocr_processor = SlideOCR()
    pdf_files_list = list(PDF_DIR.glob("*.pdf"))

    if not pdf_files_list:
        print(f"Lỗi: Không tìm thấy file PDF nào trong thư mục {PDF_DIR}.")
        print("Vui lòng kiểm tra lại đường dẫn PDF_DIR ở Cell 4.")
    else:
        print(f"Tìm thấy {len(pdf_files_list)} file PDF. Bắt đầu xử lý...")
        GLOBAL_OCR_DATA = ocr_processor.process_slides(pdf_files_list)

        # Lưu vào file JSON trên Drive
        with open(OCR_DATA_PATH, 'w', encoding='utf-8') as f:
            json.dump(GLOBAL_OCR_DATA, f, indent=2, ensure_ascii=False)

        print(f"\nOCR hoàn tất! Đã xử lý {len(GLOBAL_OCR_DATA)} trang có text.")
        print(f"Dữ liệu OCR đã được lưu vào {OCR_DATA_PATH}")

Đã tìm thấy file OCR tại: /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output/ocr_data.json. Đang tải...
Tải 177 trang từ file JSON thành công.


# Pipeline A

In [6]:
class PipelineA_BM25:
    def __init__(self):
        self.bm25_index = None
        self.ocr_data = []
        self.index_path = BM25_INDEX_PATH # Dùng đường dẫn toàn cục

    def _tokenize(self, text):
        return text.lower().split()

    def build_index(self, ocr_data: List[Dict[str, Any]], force_rebuild=False):
        if not force_rebuild and self.index_path.exists():
            print(f"Pipeline A: Đã tìm thấy index, đang tải từ {self.index_path}...")
            self.search("query") # Tải index
            print("Pipeline A: Tải index BM25 thành công.")
            return

        print("Pipeline A: Bắt đầu build index BM25...")
        self.ocr_data = ocr_data
        if not self.ocr_data:
            print("Lỗi (BM25): Không có dữ liệu OCR để build index.")
            return

        tokenized_corpus = [self._tokenize(doc['text']) for doc in self.ocr_data]
        self.bm25_index = BM25Okapi(tokenized_corpus)

        with open(self.index_path, 'wb') as f:
            pickle.dump((self.bm25_index, self.ocr_data), f)
        print(f"Pipeline A: Build index BM25 thành công, đã lưu vào {self.index_path}")

    def search(self, query: str, top_k=5) -> List[Dict[str, Any]]:
        if not self.bm25_index:
            if self.index_path.exists():
                with open(self.index_path, 'rb') as f:
                    self.bm25_index, self.ocr_data = pickle.load(f)
            else:
                print("Lỗi (BM25): Index chưa được build. Vui lòng chạy build_index() trước.")
                return []

        tokenized_query = self._tokenize(query)
        scores = self.bm25_index.get_scores(tokenized_query)
        top_n_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        results = []
        for i in top_n_indices:
            if scores[i] > 0:
                doc = self.ocr_data[i]
                results.append({
                    "source": doc['source'],
                    "page": doc['page'],
                    "score": scores[i],
                    "text_snippet": doc['text']
                })
        return results

# Pipeline C

In [7]:
class PipelineC_MiniLM:
    def __init__(self):
        self.model = None
        self.index_path = MINILM_INDEX_PATH # Dùng đường dẫn toàn cục
        self.corpus_embeddings = None
        self.ocr_data = []

    def _get_model(self):
        if self.model is None:
            print("Pipeline C: Bắt đầu tải model MiniLM-L6...")
            self.model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
            print("Pipeline C: Tải model MiniLM-L6 thành công.")

    def build_index(self, ocr_data: List[Dict[str, Any]], force_rebuild=False):
        if not force_rebuild and self.index_path.exists():
            print(f"Pipeline C: Đã tìm thấy index, đang tải từ {self.index_path}...")
            self.search("query") # Tải index
            print("Pipeline C: Tải index MiniLM thành công.")
            return

        self._get_model()
        print("Pipeline C: Bắt đầu build index MiniLM...")
        self.ocr_data = ocr_data
        if not self.ocr_data:
            print("Lỗi (MiniLM): Không có dữ liệu OCR để build index.")
            return

        corpus = [doc['text'] for doc in self.ocr_data]
        print(f"Bắt đầu embed {len(corpus)} văn bản trang...")
        self.corpus_embeddings = self.model.encode(corpus, convert_to_tensor=True, device=DEVICE, show_progress_bar=True)

        with open(self.index_path, 'wb') as f:
            pickle.dump((self.corpus_embeddings.cpu(), self.ocr_data), f)
        print(f"Pipeline C: Build index MiniLM thành công! Đã lưu vào {self.index_path}")

    def search(self, query: str, top_k=5) -> List[Dict[str, Any]]:
        self._get_model()
        if self.corpus_embeddings is None:
            if self.index_path.exists():
                with open(self.index_path, 'rb') as f:
                    self.corpus_embeddings, self.ocr_data = pickle.load(f)
                self.corpus_embeddings = self.corpus_embeddings.to(DEVICE)
            else:
                print("Lỗi (MiniLM): Index chưa được build. Vui lòng chạy build_index() trước.")
                return []

        query_embedding = self.model.encode(query, convert_to_tensor=True, device=DEVICE)
        cos_scores = st_util.pytorch_cos_sim(query_embedding, self.corpus_embeddings)[0]
        top_results = torch.topk(cos_scores, k=min(top_k, len(cos_scores)))

        results = []
        for score, idx in zip(top_results[0].cpu().tolist(), top_results[1].cpu().tolist()):
            if score > 0.1:
                doc = self.ocr_data[idx]
                results.append({
                    "source": doc['source'],
                    "page": doc['page'],
                    "score": score,
                    "text_snippet": doc['text']
                })
        return results

# Pipeline B

In [8]:
from colpali_engine.models import ColQwen2_5, ColQwen2_5_Processor

class PipelineB_ColQwen:
    def __init__(self):
        self.processor = None
        self.model = None
        self.index_path = COLQWEN_INDEX_PATH # Dùng đường dẫn toàn cục
        self.colqwen_index = [] # Chứa dicts: {'source', 'page', 'embeddings'}

    def _get_model(self):
        if self.model is None:
            print("Pipeline B: Bắt đầu tải ColQwen 2.5 model (Colpali Engine)...")
            model_name = 'vidore/colqwen2.5-v0.2'

            self.processor = ColQwen2_5_Processor.from_pretrained(model_name)
            self.model = ColQwen2_5.from_pretrained(
                model_name,
                # Dùng bfloat16 nếu có GPU, hoặc float32
                torch_dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
                device_map=DEVICE.type # Đẩy model vào GPU/CPU
            ).eval()
            print("Pipeline B: Tải ColQwen 2.5 model thành công.")

    def build_index(self, pdf_dir: Path, force_rebuild=False):
        if not force_rebuild and self.index_path.exists():
            print(f"Pipeline B: Đã tìm thấy index, đang tải từ {self.index_path}...")
            # Chỉ tải index, không cần chạy search để test
            with open(self.index_path, 'rb') as f:
                self.colqwen_index = pickle.load(f)
            print("Pipeline B: Tải index ColQwen thành công.")
            return

        self._get_model()
        print("Pipeline B: Bắt đầu build index ColQwen (Image) với Colpali Engine...")

        pdf_files = list(pdf_dir.glob("*.pdf"))
        if not pdf_files:
            print("Lỗi (ColQwen): Không tìm thấy file PDF.")
            return

        self.colqwen_index = []

        with torch.no_grad():
            for pdf_path in tqdm(pdf_files, desc="Processing PDFs (ColQwen)"):
                try:
                    images = convert_from_path(pdf_path, dpi=200)
                    for i, page_image in enumerate(tqdm(images, desc=f"Embed {pdf_path.name}", leave=False)):
                        page_num = i + 1

                        # Process image
                        image_inputs = self.processor.process_images([page_image]).to(self.model.device)

                        # Get embeddings
                        doc_outputs = self.model(**image_inputs)

                        self.colqwen_index.append({
                            'source': pdf_path.name,
                            'page': page_num,
                            'embeddings': doc_outputs.cpu() # Lưu vào CPU để tránh tràn VRAM/Dùng pickle
                        })

                except Exception as e:
                    logging.error(f"[ColQwen] Lỗi khi xử lý {pdf_path}: {e}")

        if self.colqwen_index:
            with open(self.index_path, 'wb') as f:
                pickle.dump(self.colqwen_index, f)
            print(f"Pipeline B: Build index ColQwen thành công, đã lưu vào {self.index_path}")
        else:
            print("Pipeline B: Không embed được tài liệu nào.")

    def search(self, query: str, top_k=5) -> List[Dict[str, Any]]:
        self._get_model()
        if not self.colqwen_index:
            if self.index_path.exists():
                with open(self.index_path, 'rb') as f:
                    self.colqwen_index = pickle.load(f)
            else:
                print("Lỗi (ColQwen): Index chưa được build. Vui lòng chạy build_index() trước.")
                return []

        # 1. Embed query
        query_inputs = self.processor.process_queries([query]).to(self.model.device)
        with torch.no_grad():
            query_embeds = self.model(**query_inputs)

        page_scores = []

        # 2. Score using ColBERT-style multi-vector scoring
        for doc in self.colqwen_index:
            # Chuyển embeddings về GPU để tính toán nếu đang dùng GPU
            doc_embeds = doc['embeddings'].to(self.model.device)

            # Sử dụng hàm tính điểm chuyên biệt
            score = self.processor.score_multi_vector(query_embeds, doc_embeds).item()
            page_scores.append(score)

        # 3. Get top results
        top_n_indices = sorted(range(len(page_scores)), key=lambda i: page_scores[i], reverse=True)[:top_k]

        results = []
        for i in top_n_indices:
            doc_info = self.colqwen_index[i]
            results.append({
                "source": doc_info['source'],
                "page": doc_info['page'],
                "score": page_scores[i],
                # Giữ nguyên snippet để khớp với hàm get_llm_answer
                "text_snippet": f"[ColQwen không trả về text, chỉ trả về ảnh trang {doc_info['page']}]"
            })
        return results

# Pass the retrieved context to LLM

In [9]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [10]:
if not OPENAI_API_KEY:
    print("Bạn chưa nhập API Key. Bước LLM sẽ không hoạt động.")
    client = None
else:
    client = OpenAI(api_key=OPENAI_API_KEY)

def get_image_base64_from_pdf(pdf_filename: str, page_num: int) -> str | None:
    """Tải và chuyển đổi một trang PDF cụ thể sang Base64."""
    from pdf2image import convert_from_path
    from PIL import Image
    import io
    import base64

    full_pdf_path = PDF_DIR / pdf_filename
    if not full_pdf_path.exists():
        logging.error(f"Không tìm thấy file PDF: {full_pdf_path}")
        return None

    try:
        # Lấy trang cụ thể
        images = convert_from_path(
            str(full_pdf_path),
            first_page=page_num,
            last_page=page_num,
            dpi=150 # Độ phân giải vừa phải để tiết kiệm token
        )
        if not images:
            return None

        img = images[0]
        # Chuyển ảnh PIL sang định dạng Base64
        buffered = io.BytesIO()
        img.save(buffered, format="JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")

    except Exception as e:
        logging.error(f"Lỗi khi render ảnh trang {page_num} của {pdf_filename}: {e}")
        return None

def get_llm_answer(query: str, retrieved_contexts: List[Dict[str, Any]], pipeline_type: str) -> str:
    """
    Gọi GPT-4o Mini để tổng hợp câu trả lời, hỗ trợ multi-modal cho Pipeline B.
    pipeline_type: 'A', 'B', hoặc 'C'
    """
    if client is None:
        return "(Lỗi: Chưa cấu hình OpenAI API Key)"

    if not retrieved_contexts:
        return "(Không tìm thấy ngữ cảnh nào để trả lời)"

    prompt_messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Your task is to answer the user's question *only* based on the provided context (which may include text and/or images). The response must be written in the **same language** as the user's query. If the context does not contain the information, state clearly: 'Based on the context provided, I cannot find the information'. **IMPORTANT**: ONLY INFER FROM THE GIVEN RETRIEVED CONTEXTS REGARDLESS OF HOW NONSENSE IT IS FROM THE REAL LIFE OR KNOWLEDGE YOU HAVE KNOWN. DO NOT COME UP WITH EXTERNAL FACTS OR INFORMATION."
        }
    ]
    user_content = []
    text_context_for_llm = ""

    for i, ctx in enumerate(retrieved_contexts):
        source_info = f"Source: {ctx['source']}, Page: {ctx['page']}, Score: {ctx['score']:.4f}"

        if pipeline_type in ['A', 'C']:
            # Pipeline A và C (Text-only): Chỉ thêm text
            snippet = ctx['text_snippet'][:1000] + "..."
            text_context_for_llm += f"--- Context {i+1} ({source_info}) ---\n"
            text_context_for_llm += f"Text Snippet:\n{snippet}\n\n"

        elif pipeline_type == 'B':
            # Pipeline B (Multi-modal): Thêm ảnh và text ngắn chỉ dẫn
            base64_image = get_image_base64_from_pdf(ctx['source'], ctx['page'])
            if base64_image:
                # 1. Thêm ảnh vào user_content
                user_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}})
                # 2. Thêm text ngắn gọn để LLM xác định ảnh nào là ngữ cảnh nào
                text_context_for_llm += f"Context {i+1}: Image is provided above. ({source_info})\n\n"

    # Text prompt cuối cùng
    final_prompt = f"**Contexts from Retriever:**\n{text_context_for_llm}\n\n**Question:**\n{query}"

    # Đặt text prompt lên đầu mảng user_content (cần thiết cho mô hình)
    user_content.insert(0, {"type": "text", "text": final_prompt})

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini", # Hỗ trợ multi-modal
            messages=[{"role": "user", "content": user_content}],
            temperature=0.1,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"(Lỗi khi gọi LLM: {str(e)})"

# Load model and index

In [11]:
# -----------------------------------------------------------------
# Đặt cờ này thành True nếu bạn muốn build lại TẤT CẢ index
# (ví dụ: khi bạn thêm/xóa file PDF)
# -----------------------------------------------------------------
FORCE_REBUILD_ALL = False
# -----------------------------------------------------------------


# --- Tải dữ liệu OCR (nếu chưa có) ---
if not GLOBAL_OCR_DATA:
    print("GLOBAL_OCR_DATA trống. Đang kiểm tra file...")
    if OCR_DATA_PATH.exists():
        with open(OCR_DATA_PATH, 'r', encoding='utf-8') as f:
            GLOBAL_OCR_DATA = json.load(f)
        print(f"Đã tải {len(GLOBAL_OCR_DATA)} trang từ file OCR.")
    else:
        print("LỖI NGHIÊM TRỌNG: Không có dữ liệu OCR.")
        print("Vui lòng chạy lại Cell 6 để tạo dữ liệu OCR trước.")

if GLOBAL_OCR_DATA:
    p_a = PipelineA_BM25()
    p_b = PipelineB_ColQwen()
    p_c = PipelineC_MiniLM()

    print("--- Bắt đầu Build/Load Index (việc này có thể mất thời gian) ---")

    p_a.build_index(GLOBAL_OCR_DATA, force_rebuild=FORCE_REBUILD_ALL)
    p_b.build_index(PDF_DIR, force_rebuild=FORCE_REBUILD_ALL)
    p_c.build_index(GLOBAL_OCR_DATA, force_rebuild=FORCE_REBUILD_ALL)

    print("HOÀN TẤT BUILD/LOAD TẤT CẢ 3 INDEX.")

--- Bắt đầu Build/Load Index (việc này có thể mất thời gian) ---
Pipeline A: Đã tìm thấy index, đang tải từ /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output/bm25_index.pkl...
Pipeline A: Tải index BM25 thành công.
Pipeline B: Đã tìm thấy index, đang tải từ /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output/colqwen_index.pkl...
Pipeline B: Tải index ColQwen thành công.
Pipeline C: Đã tìm thấy index, đang tải từ /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output/minilm_index.pkl...
Pipeline C: Bắt đầu tải model MiniLM-L6...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pipeline C: Tải model MiniLM-L6 thành công.
Pipeline C: Tải index MiniLM thành công.
HOÀN TẤT BUILD/LOAD TẤT CẢ 3 INDEX.


# Run pipelines

In [12]:
from IPython.display import display, Markdown

queries = [
    # "What is the plane equation used in game physics?",
    # "What is the difference between AABB and bounding sphere collision detection?",
    # "What is Euler integration used for in game physics?",
    # "How is the dot product used in collision detection?",
    # "What is swept sphere intersection?",
    # "What is a sprite sheet?",
    # "What is parallax scrolling?",
    # "What are tile maps used for?",
    # "What are the advantages of PNG and TGA image formats in 2D games?",
    # "What is the clamp algorithm in scrolling games?",
    # "What is the A* algorithm in game AI?",
    # "Give me an example of Greedy Best First Search.",
    # "What is Dijkstra’s algorithm used for?",
    # "What is a behavior tree in game AI?",
    # "What is the minimax algorithm?",
    # "What are alpha-beta cutoffs?",
    # "What is MCTS and what are its advantages compare to Minimax?",
    # "Describe an example of alpha-beta cutoffs.",
    # "What is client prediction in networked games?",
    # "What is the difference between TCP and UDP in games?",
    # "What is peer-to-peer topology?",
    # "What is cheating in online games?",
    # "What are homogeneous coordinates in 3D graphics?",
    # "What is the difference between orthographic and perspective projection?",
    # "What are quaternions used for in 3D graphics?",
    # "What is Z-buffering?",
    # "What are transform matrices used for?",
    # "What is the game loop?",
    # "What are the major eras of video game evolution?",
    # "What are sprites and why are they used?",
    # "How does pygame draw basic shapes?"
]
# -----------------------------------------------------------------

if 'OUTPUT_DIR' not in locals():
    print("Lỗi: OUTPUT_DIR chưa được định nghĩa. Vui lòng chạy Cell 4.")
    # Tạo một thư mục tạm nếu chưa có
    OUTPUT_DIR = Path("./temp_output")
    OUTPUT_DIR.mkdir(exist_ok=True)
    print(f"Đã tạo thư mục tạm: {OUTPUT_DIR}")

# Hàm tiện ích để chuẩn hóa query thành tên file
def sanitize_filename(query):
    # Thay thế các ký tự không an toàn bằng dấu gạch dưới
    filename = re.sub(r'[^a-zA-Z0-9\s-]', '', query).lower()
    # Thay thế khoảng trắng và dấu gạch nối bằng dấu gạch dưới
    filename = re.sub(r'[_\s-]+', '_', filename).strip('_')
    # Giới hạn độ dài tên file
    return filename[:50]

for query in queries:
    # 1. Tạo tên file và đường dẫn
    filename = sanitize_filename(query)
    output_path = OUTPUT_DIR / f"{filename}.md"
    markdown_content = f"# RAG Comparison Results for: '{query}'\n\n"
    print(f"\n--- Đang thực hiện truy vấn: '{query}' ---")

    # --- Chạy Pipeline A ---
    markdown_content += "\n# Pipeline A (OCR + BM25)\n"
    results_a = p_a.search(query)

    markdown_content += "## 🔍 Retrieved Contexts\n"
    for res in results_a:
        info_a = f"Nguồn: {res['source']}, Trang: {res['page']} (Score: {res['score']:.2f})\n"
        print(info_a.strip())
        markdown_content += f"* {info_a} \n"

    llm_answer_a = get_llm_answer(query, results_a, 'A')
    markdown_content += "\n## 🤖 LLM Summary (Text-only):\n"
    markdown_content += f"> {llm_answer_a}\n"
    print(f"\nPipeline A hoàn tất.\n")


    # --- Chạy Pipeline B (Multi-modal) ---
    markdown_content += "\n# Pipeline B (ColQwen - Multi-modal)\n"
    results_b = p_b.search(query)

    markdown_content += "## 🔍 Retrieved Contexts\n"
    for res in results_b:
        info_b = f"Nguồn: {res['source']}, Trang: {res['page']} (Score: {res['score']:.4f})"
        print(info_b)
        markdown_content += f"* {info_b} (Image sent to LLM)\n"

    llm_answer_b = get_llm_answer(query, results_b, 'B')
    markdown_content += "\n## 🤖 LLM Summary (Multi-modal):\n"
    markdown_content += f"> {llm_answer_b}\n"
    print(f"\nPipeline B hoàn tất.\n")


    # --- Chạy Pipeline C ---
    markdown_content += "\n# Pipeline C (OCR + MiniLM-L6)\n"
    results_c = p_c.search(query)

    markdown_content += "## 🔍 Retrieved Contexts\n"
    for res in results_c:
        info_c = f"Nguồn: {res['source']}, Trang: {res['page']} (Score: {res['score']:.4f})"
        print(info_c)
        markdown_content += f"* {info_c} \n"

    llm_answer_c = get_llm_answer(query, results_c, 'C')
    markdown_content += "\n## 🤖 LLM Summary (Text-only):\n"
    markdown_content += f"> {llm_answer_c}\n"
    print(f"\nPipeline C hoàn tất.\n")

    # 2. Lưu nội dung Markdown vào file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(markdown_content)

    print(f"\n✨ Đã lưu kết quả của truy vấn '{query}' vào: {output_path}")
    print("\n" + "="*80) # Dùng dấu bằng dài hơn để dễ phân biệt các truy vấn


--- Đang thực hiện truy vấn: 'What is the plane equation used in game physics?' ---
Nguồn: Chap_1_Math_Physic.pdf, Trang: 26 (Score: 10.94)
Nguồn: Chap_0_Introduction.pdf, Trang: 6 (Score: 10.22)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 37 (Score: 9.41)
Nguồn: Chap4_AI_Minimax.pdf, Trang: 9 (Score: 7.84)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 28 (Score: 7.62)

Pipeline A hoàn tất.

Pipeline B: Bắt đầu tải ColQwen 2.5 model (Colpali Engine)...


preprocessor_config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/240M [00:00<?, ?B/s]

Pipeline B: Tải ColQwen 2.5 model thành công.
Nguồn: Chap_1_Math_Physic.pdf, Trang: 25 (Score: 15.2500)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 26 (Score: 15.0625)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 49 (Score: 15.0625)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 2 (Score: 14.6875)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 1 (Score: 13.8750)

Pipeline B hoàn tất.

Nguồn: Chap_1_Math_Physic.pdf, Trang: 25 (Score: 0.5353)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 1 (Score: 0.5044)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 26 (Score: 0.4791)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 49 (Score: 0.4614)
Nguồn: Chap_1_Math_Physic.pdf, Trang: 2 (Score: 0.4586)

Pipeline C hoàn tất.


✨ Đã lưu kết quả của truy vấn 'What is the plane equation used in game physics?' vào: /content/drive/MyDrive/CapstoneProject/Retriever/RAG_Output/what_is_the_plane_equation_used_in_game_physics.md


--- Đang thực hiện truy vấn: 'What is client prediction in networked games?' ---
Nguồn: Chap_4_Network.pdf, Trang: 10 (Score: 11.35)
Nguồ

## Top 1: Colpali / ColQwen (Pipeline B)
- Lý do: Đây là mô hình multi-modal. Các file slide không chỉ chứa văn bản mà còn chứa hình ảnh, sơ đồ, bố cục (layout), và văn bản trong các hình ảnh đó.

- Ưu điểm:
    - Colpali "nhìn" được toàn bộ trang slide như một con người, cho phép nó hiểu được ngữ cảnh từ bố cục và các yếu tố phi văn bản.
    - Nó không bị phụ thuộc vào chất lượng của OCR. Nếu OCR nhận dạng sai hoặc bỏ sót văn bản (ví dụ: text trong một sơ đồ phức tạp), Colpali vẫn có thể tìm thấy trang đó nếu hình ảnh của trang khớp với query.

- Nhược điểm:
    - Nặng, tốn tài nguyên (cần GPU) và thời gian build index lâu hơn.
    - Nó retrieve toàn bộ trang (dưới dạng ảnh), chứ không phải đoạn văn bản cụ thể. (Đây là lý do chúng ta phải thêm bước "lấy text OCR" cho nó trước khi đưa vào LLM).

## Top 2: OCR + MiniLM-L6 (Pipeline C - Mới)
- Lý do: Đây là phương pháp semantic search (tìm kiếm ngữ nghĩa).
- Ưu điểm: MiniLM hiểu được ý nghĩa và ngữ cảnh của câu query, chứ không chỉ là từ khóa.
- Nhược điểm: Phụ thuộc hoàn toàn vào OCR: Nếu OCR chạy sai, thiếu chữ, hoặc không đọc được text, pipeline này sẽ thất bại (rác vào -> rác ra). Nó không thể "nhìn" thấy sơ đồ hay hình ảnh.

## Top 3: OCR + BM25 (Pipeline A)
- Lý do: Đây là phương pháp lexical search (tìm kiếm từ vựng/từ khóa).
- Ưu điểm: Rất nhanh, nhẹ và cực kỳ hiệu quả nếu người dùng tìm kiếm chính xác từ khóa có trong tài liệu (ví dụ: tìm một tên riêng, một thuật ngữ cụ thể).

- Nhược điểm:
    - Cũng phụ thuộc hoàn toàn vào OCR.
    - Không hiểu ngữ nghĩa. Nếu bạn tìm "chi phí vận hành" và văn bản ghi "operating costs", BM25 sẽ không tìm thấy. Nó cũng không hiểu được từ đồng nghĩa hay các cách diễn đạt khác nhau.

## Kết luận
Colpali (ColQwen) tốt nhất vì nó hiểu được tính trực quan của slide. MiniLM tốt nhì vì nó hiểu ngữ nghĩa của văn bản. BM25 xếp cuối vì nó chỉ khớp từ khóa. (Tuy nhiên, một hệ thống RAG tốt nhất thường kết hợp cả BM25 và MiniLM - gọi là "hybrid search" - để lấy ưu điểm của cả hai).